In [6]:
# ==========================================
# CELL 1: PYTORCH 2.6.0 (cu126) + TRITON 3.3.1
# (không import torch trong kernel này)
# ==========================================
import subprocess, sys

def probe(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

TORCH_PROBE = "import torch;print(torch.__version__, int(torch._C._GLIBCXX_USE_CXX11_ABI))"
cur = probe(TORCH_PROBE)
print("Torch hiện tại:", cur)

if cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"):
    print("⏭️ Đã có torch 2.6.0+cu126 (ABI=True), bỏ qua bước cài torch.")
else:
    print("🧹 Dọn dẹp và cài lại PyTorch...")
    !rm -rf /usr/local/lib/python3.13/dist-packages/~*
    !pip uninstall torch torchvision torchaudio causal-conv1d mamba-ssm -y -q
    !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126 -q
    cur = probe(TORCH_PROBE)
    print("Torch sau khi cài:", cur)

assert cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"), \
    f"❌ Torch chưa đúng (cần 2.6.0+cu126 và ABI=1, đang là: {cur}). Dừng lại và gửi mình dòng này."

print("\n⚙️ Nâng Triton (torch kéo theo 3.2.0, mamba_ssm cần bản mới hơn)...")
!python -m pip install triton==3.3.1 -q
trt = probe("import triton;print(triton.__version__, hasattr(triton,'set_allocator'))")
print("Triton:", trt)
assert trt and trt.endswith("True"), f"❌ Triton chưa có set_allocator (đang là: {trt})"

print("\n✅ CELL 1 XONG. Chạy tiếp cell 2.")

Torch hiện tại: 2.6.0+cu126 1
⏭️ Đã có torch 2.6.0+cu126 (ABI=True), bỏ qua bước cài torch.

⚙️ Nâng Triton (torch kéo theo 3.2.0, mamba_ssm cần bản mới hơn)...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamba-ssm 2.3.2.post1 requires apache-tvm-ffi<=0.1.9, which is not installed.
mamba-ssm 2.3.2.post1 requires ninja, which is not installed.
mamba-ssm 2.3.2.post1 requires quack-kernels>=0.3.4, which is not installed.
mamba-ssm 2.3.2.post1 requires tilelang==0.1.8, which is not installed.
torch 2.6.0+cu126 requires triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 3.3.1 which is incompatible.
mamba-ssm 2.3.2.post1 requires triton>=3.5.0, but you have triton 3.3.1 which is incompatible.
Triton: 3.3.1 True

✅ CELL 1 XONG. Chạy tiếp cell 2.


In [7]:
# ==========================================
# CELL 2: MOUNT DRIVE + CÀI 2 WHEEL (chọn theo ABI thật)
# (vẫn không import torch trong kernel này)
# ==========================================
import os, subprocess, sys
from google.colab import drive

print("🔗 Đang kết nối Google Drive...")
drive.mount('/content/drive')

WHEEL_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels"

r = subprocess.run([sys.executable, "-c",
                    "import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')"],
                   capture_output=True, text=True)
abi = r.stdout.strip()
assert abi in ("TRUE", "FALSE"), f"❌ Không đọc được ABI: {r.stderr}"
print(f"🔍 ABI của torch đã cài: cxx11abi{abi}")

CAUSAL_WHL = f"{WHEEL_DIR}/causal_conv1d-1.7.0+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"
MAMBA_WHL  = f"{WHEEL_DIR}/mamba_ssm-2.3.2.post1+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"

for p in (CAUSAL_WHL, MAMBA_WHL):
    assert os.path.exists(p), f"❌ Không thấy wheel: {p}"

print("\n🚀 Đang cài 2 wheel từ Drive...")
!python -m pip install "{CAUSAL_WHL}" --force-reinstall --no-deps -q
!python -m pip install "{MAMBA_WHL}" --force-reinstall --no-deps -q

print("\n✅ CELL 2 XONG. Chạy tiếp cell 3 (khởi động lại runtime).")

🔗 Đang kết nối Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 ABI của torch đã cài: cxx11abiTRUE

🚀 Đang cài 2 wheel từ Drive...

✅ CELL 2 XONG. Chạy tiếp cell 3 (khởi động lại runtime).


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# ==========================================
# CELL 4: KIỂM TRA causal_conv1d + mamba_ssm (xuôi + ngược, fp32 + fp16)
# ==========================================
import sys, traceback, torch

print("python :", f"cp{sys.version_info.major}{sys.version_info.minor}")
print("torch  :", torch.__version__, "| ABI =", torch._C._GLIBCXX_USE_CXX11_ABI)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
import triton
print("triton :", triton.__version__, "| set_allocator:", hasattr(triton, "set_allocator"))

ok = {}

# --- Causal-Conv1d ---
try:
    import causal_conv1d, causal_conv1d_cuda
    from causal_conv1d import causal_conv1d_fn
    for dt in (torch.float32, torch.float16):
        x = torch.randn(2, 16, 64, device="cuda", dtype=dt, requires_grad=True)
        w = torch.randn(16, 4, device="cuda", dtype=dt, requires_grad=True)
        y = causal_conv1d_fn(x, w, activation="silu")
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(x.grad).all()
    ok["causal_conv1d"] = True
    print("causal_conv1d : ✅ xuôi + ngược, fp32 + fp16")
except Exception:
    ok["causal_conv1d"] = False
    print("causal_conv1d : ❌"); traceback.print_exc()

# --- Mamba-SSM ---
try:
    import mamba_ssm, selective_scan_cuda
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    B, D, L, N = 2, 16, 64, 8
    for dt in (torch.float32, torch.float16):
        u     = torch.randn(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        delta = torch.rand(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        A     = (-torch.rand(D, N, device="cuda")).requires_grad_()
        Bm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Cm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Dv    = torch.randn(D, device="cuda")
        y = selective_scan_fn(u, delta, A, Bm, Cm, D=Dv, delta_softplus=True)
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(u.grad).all()
    ok["mamba_ssm"] = True
    print("mamba_ssm     : ✅ selective_scan xuôi + ngược, fp32 + fp16")
except Exception:
    ok["mamba_ssm"] = False
    print("mamba_ssm     : ❌"); traceback.print_exc()

print("\n==>", "✅ SẴN SÀNG" if all(ok.values()) else "❌ CHƯA ĐẠT, gửi mình toàn bộ log ở trên")

python : cp313
torch  : 2.6.0+cu126 | ABI = True
GPU    : Tesla T4
triton : 3.3.1 | set_allocator: True
causal_conv1d : ✅ xuôi + ngược, fp32 + fp16
mamba_ssm     : ✅ selective_scan xuôi + ngược, fp32 + fp16

==> ✅ SẴN SÀNG


In [3]:
# ==========================================
# CELL 5: VÀO THƯ MỤC DỰ ÁN + SELFTEST MAMBA-UNET
# ==========================================
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES"
os.chdir(PROJECT_DIR)
print("📁 Đang làm việc tại:", os.getcwd())

!python -m mamba_unet.train_main selftest

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Đang làm việc tại: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES
🧪 Bắt đầu selftest toàn diện mamba_unet.train_main...
[2026-11-10 08:53:53] Epoch [01/04] | LR: 1.00e-02 | Train [Loss: 0.5303, Dice: 0.0000] | Val [Loss: 0.5301, Dice: 0.0000] | Time: 00m 02s | ETA: 00m 05s ⭐ [BEST]
[2026-11-10 08:53:53] Epoch [02/04] | LR: 1.00e-02 | Train [Loss: 0.5292, Dice: 0.0000] | Val [Loss: 0.5302, Dice: 0.0000] | Time: 00m 00s | ETA: 00m 02s 
[2026-11-10 08:53:53] 
⚠️ ĐÃ NHẬN TÍN HIỆU DỪNG (KeyboardInterrupt)!
   - Checkpoints đã được bảo toàn an toàn tại: /tmp/tmpqxmobsyu/selftest_run
   - Hoàn tất an toàn tới hết Epoch 2.
[2026-11-10 08:53:53] [FINISH] Phiên chạy 'selftest_run' kết thúc với trạng thái: INTERRUPTED
🔄 RESUME THÀNH CÔNG từ 'last_model.pth' | Đã xong epoch 2 -> Tiếp tục từ epoch 3 | Best Val Dice: 0.0000
[2026-11-10 08:53:

In [4]:
import os
os.chdir("/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES")

# 1) Ép backend CUDA: nếu kernel hỏng thì lỗi ngay thay vì âm thầm chạy chậm
!DUNEAI_MAMBA_BACKEND=cuda python -m mamba_unet.train_main selftest

# 2) Mô hình mamba_vision_T đầy đủ (phải báo 367/367 khóa)
!python -m mamba_unet.model

# 3) Kiểm thử đầy đủ của dự án
!python tests/run_all.py

🧪 Bắt đầu selftest toàn diện mamba_unet.train_main...
[2026-11-10 08:55:08] Epoch [01/04] | LR: 1.00e-02 | Train [Loss: 0.5313, Dice: 0.0000] | Val [Loss: 0.5302, Dice: 0.0000] | Time: 00m 01s | ETA: 00m 02s ⭐ [BEST]
[2026-11-10 08:55:08] Epoch [02/04] | LR: 1.00e-02 | Train [Loss: 0.5306, Dice: 0.0000] | Val [Loss: 0.5302, Dice: 0.0000] | Time: 00m 00s | ETA: 00m 01s 
[2026-11-10 08:55:08] 
⚠️ ĐÃ NHẬN TÍN HIỆU DỪNG (KeyboardInterrupt)!
   - Checkpoints đã được bảo toàn an toàn tại: /tmp/tmp0ge_69v_/selftest_run
   - Hoàn tất an toàn tới hết Epoch 2.
[2026-11-10 08:55:08] [FINISH] Phiên chạy 'selftest_run' kết thúc với trạng thái: INTERRUPTED
🔄 RESUME THÀNH CÔNG từ 'last_model.pth' | Đã xong epoch 2 -> Tiếp tục từ epoch 3 | Best Val Dice: 0.0000
[2026-11-10 08:55:09] Epoch [03/04] | LR: 1.00e-02 | Train [Loss: 0.5299, Dice: 0.0000] | Val [Loss: 0.5303, Dice: 0.0000] | Time: 00m 00s | ETA: 00m 00s 
[2026-11-10 08:55:09] Epoch [04/04] | LR: 1.00e-02 | Train [Loss: 0.5299, Dice: 0.0000] |